# DASE7506 MP1 - Small Language Model Challenge (Kaggle runner)

This notebook trains and evaluates the improved model on the supplied WikiText-2
data. Before running:

1. Create a Kaggle **Dataset** from `mp1_data.zip` (it extracts to `code/...`).
2. Create a new Notebook, **Add Input -> your dataset**, and select a GPU
   (Settings -> Accelerator -> GPU T4 x2 or P100).
3. Set `MODE` in the *Experiment list* cell to `"smoke"` for a first check, then
   `"full"` for the complete suite.
4. Run all cells. Results are printed at the end and saved to
   `/kaggle/working/mp1_results.zip` for download.

The supplied data and evaluator are never modified; only `student.py`,
`train_student.py`, `suite.py` and the config are written here.


In [ ]:
import os, sys, glob, shutil, subprocess, pathlib, json, time
started = time.time()

hits = glob.glob('/kaggle/input/**/data/tokenizer.json', recursive=True)
assert hits, 'No dataset containing code/data/tokenizer.json was found under /kaggle/input.'
src_code = pathlib.Path(hits[0]).parent.parent
print('dataset code dir:', src_code)

dst_code = pathlib.Path('/kaggle/working/code')
if dst_code.exists():
    shutil.rmtree(dst_code)
shutil.copytree(src_code, dst_code)
os.chdir(dst_code)
print('working dir:', os.getcwd())

try:
    import tokenizers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers==0.21.4'], check=True)

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
    print('memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
%%writefile /kaggle/working/code/student.py
"""MP1 student implementation.

A compute-matched, modernized decoder-only transformer. Every design choice is
a config flag so the same code path can produce the baseline-style control and
the full model, which keeps the ablations clean:

    config['pos_emb']  'rope' (default) | 'learned'
    config['mlp']      'swiglu' (default) | 'gelu'
    config['norm']     'rmsnorm' (default) | 'layernorm'
    config['qk_norm']  bool (default False)
    config['tied']     bool (default True)

The model exposes the two interfaces required by the classroom pipeline:
``forward(ids)`` returns unnormalized logits and ``predict_log_probs(ids)``
returns finite, normalized natural-log probabilities. Both are strictly causal
and keep no state across calls, so independent 256-token windows start fresh.

``build_model(config)`` is the factory used by ``train.py`` / ``evaluate.py``.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F


class RMSNorm(nn.Module):
    """Root-mean-square normalization (no mean subtraction, no bias)."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * x * scale


class LayerNorm(nn.Module):
    """Standard LayerNorm without bias (used for the RMSNorm ablation)."""

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return F.layer_norm(x, (x.shape[-1],), self.weight, None, self.eps)


def make_norm(kind, dim):
    return RMSNorm(dim) if kind == 'rmsnorm' else LayerNorm(dim)


def build_rope_cache(head_dim, length, base=10000.0):
    """Precompute cos/sin tables of shape [length, head_dim]."""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(length).float()
    freqs = torch.outer(positions, inv_freq)          # [length, head_dim/2]
    emb = torch.cat((freqs, freqs), dim=-1)           # [length, head_dim]
    return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    """x: [batch, heads, length, head_dim]; cos/sin: [length, head_dim]."""
    return x * cos + rotate_half(x) * sin


class Attention(nn.Module):
    def __init__(self, config, width, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.proj = nn.Linear(width, width, bias=False)
        self.qk_norm = bool(config.get('qk_norm', False))
        self.q_norm = RMSNorm(self.head_dim) if self.qk_norm else None
        self.k_norm = RMSNorm(self.head_dim) if self.qk_norm else None
        self.rope = config.get('pos_emb', 'rope') == 'rope'
        self.dropout = float(config.get('dropout', 0.0))

    def forward(self, x, cos, sin):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        if self.q_norm is not None:
            q = self.q_norm(q)
            k = self.k_norm(k)
        if self.rope:
            cos, sin = cos[:length], sin[:length]
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                                  dropout_p=self.dropout if self.training else 0.0)
        return self.proj(attended.transpose(1, 2).reshape(batch, length, width))


class SwiGLU(nn.Module):
    def __init__(self, width, hidden, dropout):
        super().__init__()
        self.up = nn.Linear(width, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, width, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, b = self.up(x).chunk(2, dim=-1)
        return self.dropout(self.down(F.silu(a) * b))


class GELUMLP(nn.Module):
    def __init__(self, width, hidden, dropout):
        super().__init__()
        self.fc = nn.Linear(width, hidden, bias=False)
        self.proj = nn.Linear(hidden, width, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, config, width, heads, hidden):
        super().__init__()
        norm = config.get('norm', 'rmsnorm')
        dropout = float(config.get('dropout', 0.0))
        self.norm1, self.norm2 = make_norm(norm, width), make_norm(norm, width)
        self.attn = Attention(config, width, heads)
        mlp = config.get('mlp', 'swiglu')
        self.mlp = SwiGLU(width, hidden, dropout) if mlp == 'swiglu' else GELUMLP(width, hidden, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.mlp(self.norm2(x))


class ModernGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.context = int(config['context'])
        width = int(config['width'])
        heads = int(config['heads'])
        depth = int(config['depth'])
        vocab = int(config['vocab'])
        dropout = float(config.get('dropout', 0.0))
        hidden = int(config.get('mlp_hidden', 4 * width))
        base = float(config.get('rope_base', 10000.0))
        self.rope = config.get('pos_emb', 'rope') == 'rope'

        self.token = nn.Embedding(vocab, width)
        if not self.rope:
            self.pos = nn.Embedding(self.context, width)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(config, width, heads, hidden) for _ in range(depth)])
        self.norm = make_norm(config.get('norm', 'rmsnorm'), width)
        self.head = nn.Linear(width, vocab, bias=False)

        head_dim = width // heads
        cos, sin = build_rope_cache(head_dim, self.context, base)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

        self.apply(self._init_weights)
        for name, param in self.named_parameters():
            if name.endswith('proj.weight') or name.endswith('down.weight'):
                nn.init.normal_(param, std=0.02 / math.sqrt(2 * depth))
        if config.get('tied', True):
            self.head.weight = self.token.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def features(self, ids):
        x = self.token(ids)
        if not self.rope:
            positions = torch.arange(ids.shape[1], device=ids.device)
            x = x + self.pos(positions)
        x = self.drop(x)
        cos, sin = self.rope_cos.to(ids.device), self.rope_sin.to(ids.device)
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.norm(x)

    def forward(self, ids):
        """Training interface: unnormalized next-token logits [batch, time, vocab]."""
        return self.head(self.features(ids))

    def predict_log_probs(self, ids):
        """Evaluation interface: normalized log probabilities [batch, time, vocab]."""
        return F.log_softmax(self(ids).float(), dim=-1)


def build_model(config):
    return ModernGPT(config)


In [ ]:
%%writefile /kaggle/working/code/train_student.py
"""Improved training recipe for the MP1 student model.

Differences from the classroom ``train.py``:
    - configurable architecture config (defaults to ``configs/student.json``)
    - decoupled weight decay (no decay on norms / embeddings)
    - longer linear warmup and cosine decay to a configurable floor
    - optional exponential moving average (EMA) of the weights
    - periodic validation scoring for both raw and EMA weights
    - the checkpoint written as ``checkpoint.pt`` holds the state that scored
      best on validation, so the frozen predictor is selected without touching
      the test split.

Usage:
    python train_student.py --run-dir runs/student --steps 20000
"""
import argparse
import copy
import json
import math
import time
from pathlib import Path

import torch
from torch.nn import functional as F

from common import (PROTOCOL, ROOT, autocast, device_metrics, load_data,
                    make_model, setup, sha)
from evaluate import score


class EMA:
    """Exponential moving average of floating point model parameters."""

    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            if s.dtype.is_floating_point:
                s.mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                s.copy_(v)

    def state_dict(self):
        return self.shadow


def make_optimizer(model, lr, weight_decay, betas):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay if param.dim() >= 2 else no_decay).append(param)
    groups = [{'params': decay, 'weight_decay': weight_decay},
              {'params': no_decay, 'weight_decay': 0.0}]
    return torch.optim.AdamW(groups, lr=lr, betas=betas, eps=1e-8)


def lr_at(step, steps, base_lr, min_ratio, warmup):
    if step < warmup:
        return base_lr * (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, steps - warmup)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * (min_ratio + (1.0 - min_ratio) * coeff)


@torch.no_grad()
def evaluate_state(model, state_dict, data, device, split='validation'):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state_dict)
    result = score(model, *data[split], device, 'fp32')
    result.pop('window_nll_nats', None)
    model.load_state_dict(backup)
    return result


def consider(best, source, step, state, bpb):
    """Keep the validation-best state seen so far (checkpoint selection / early stop)."""
    if bpb < best['bpb']:
        return {'bpb': bpb, 'step': step, 'source': source,
                'state': {k: v.detach().clone() for k, v in state.items()}}
    return best


def main():
    total_started = time.perf_counter()
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--implementation', default='student')
    p.add_argument('--config', type=Path, default=ROOT / 'configs/student.json')
    p.add_argument('--run-dir', type=Path, default=ROOT / 'runs/student')
    p.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--precision', choices=['auto', 'fp32', 'bf16'], default='auto')
    p.add_argument('--threads', type=int, default=4)
    p.add_argument('--seed', type=int, default=17)
    p.add_argument('--steps', type=int, default=20000)
    p.add_argument('--batch-size', type=int, default=32)
    p.add_argument('--lr', type=float, default=1e-3)
    p.add_argument('--min-lr-ratio', type=float, default=0.1)
    p.add_argument('--warmup', type=int, default=500)
    p.add_argument('--weight-decay', type=float, default=0.1)
    p.add_argument('--beta1', type=float, default=0.9)
    p.add_argument('--beta2', type=float, default=0.95)
    p.add_argument('--grad-clip', type=float, default=1.0)
    p.add_argument('--ema-decay', type=float, default=0.999)
    p.add_argument('--no-ema', action='store_true')
    p.add_argument('--eval-every', type=int, default=1000)
    p.add_argument('--log-every', type=int, default=100)
    p.add_argument('--compile', action='store_true')
    args = p.parse_args()
    if args.steps < 1 or args.batch_size < 1:
        p.error('Batch size and step count must be positive.')
    if args.run_dir.exists() and any(args.run_dir.iterdir()):
        p.error('Run directory already contains results. Use a new --run-dir.')

    device, precision = setup(args.device, args.precision, args.threads)
    torch.manual_seed(args.seed)
    prepared = time.perf_counter()
    data = load_data()
    config = json.loads(args.config.read_text(encoding='utf-8-sig'))
    model, implementation_sha = make_model(args.implementation, config, device)
    args.run_dir.mkdir(parents=True, exist_ok=True)
    optimizer = make_optimizer(model, args.lr, args.weight_decay, (args.beta1, args.beta2))
    ema = None if args.no_ema else EMA(model, args.ema_decay)
    run_model = torch.compile(model) if args.compile else model
    tokens = data['train'][0].to(device)
    rng = torch.Generator().manual_seed(args.seed)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    preparation_seconds = time.perf_counter() - prepared
    started = time.perf_counter()

    history, validation_history = [], []
    intermediate_validation_seconds = 0.0
    best = {'bpb': float('inf'), 'step': 0, 'source': None, 'state': None}
    for step in range(args.steps):
        starts = torch.randint(len(tokens) - 257, (args.batch_size,), generator=rng).to(device)
        batch = tokens[starts[:, None] + torch.arange(257, device=device)]
        lr = lr_at(step, args.steps, args.lr, args.min_lr_ratio, args.warmup)
        for group in optimizer.param_groups:
            group['lr'] = lr
        optimizer.zero_grad(set_to_none=True)
        with autocast(device, precision):
            loss = F.cross_entropy(run_model(batch[:, :-1]).flatten(0, 1).float(), batch[:, 1:].flatten())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        optimizer.step()
        if ema is not None:
            ema.update(model)
        if (step + 1) % args.log_every == 0 or step + 1 == args.steps:
            row = {'step': step + 1, 'loss': loss.item(), 'lr': lr,
                   'seconds': time.perf_counter() - started - intermediate_validation_seconds}
            history.append(row)
            print(json.dumps(row), flush=True)
        if args.eval_every > 0 and (step + 1) % args.eval_every == 0:
            raw = evaluate_state(model, model.state_dict(), data, device)
            intermediate_validation_seconds += raw['seconds']
            entry = {'step': step + 1, 'raw': raw}
            best = consider(best, 'raw', step + 1, model.state_dict(), raw['bpb'])
            if ema is not None:
                ema_res = evaluate_state(model, ema.state_dict(), data, device)
                intermediate_validation_seconds += ema_res['seconds']
                entry['ema'] = ema_res
                best = consider(best, 'ema', step + 1, ema.state_dict(), ema_res['bpb'])
            validation_history.append(entry)
            print(json.dumps({'validation': entry}), flush=True)

    raw_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    candidates = {'raw': raw_state}
    if ema is not None:
        candidates['ema'] = ema.state_dict()
    for source, state in candidates.items():
        result = evaluate_state(model, state, data, device)
        intermediate_validation_seconds += result['seconds']
        best = consider(best, source, args.steps, state, result['bpb'])
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    train_seconds = time.perf_counter() - started - intermediate_validation_seconds

    checkpoint = args.run_dir / 'checkpoint.pt'
    torch.save({'protocol': PROTOCOL, 'implementation': args.implementation, 'config': config,
                'model': best['state'], 'seed': args.seed,
                'train_tokens': args.steps * args.batch_size * 256}, checkpoint)
    torch.save({'protocol': PROTOCOL, 'implementation': args.implementation, 'config': config,
                'model': raw_state, 'seed': args.seed,
                'train_tokens': args.steps * args.batch_size * 256}, args.run_dir / 'checkpoint_raw.pt')
    if ema is not None:
        torch.save({'protocol': PROTOCOL, 'implementation': args.implementation, 'config': config,
                    'model': ema.state_dict(), 'seed': args.seed,
                    'train_tokens': args.steps * args.batch_size * 256}, args.run_dir / 'checkpoint_ema.pt')

    result = {'protocol': PROTOCOL, 'implementation': args.implementation, 'config': config,
              'seed': args.seed, 'parameters': sum(p.numel() for p in model.parameters()),
              'precision': precision, 'train_tokens': args.steps * args.batch_size * 256,
              'preparation_seconds': preparation_seconds, 'train_seconds': train_seconds,
              'selected': {'bpb': best['bpb'], 'source': best['source'], 'step': best['step']},
              'history': history, 'validation_history': validation_history,
              'intermediate_validation_seconds': intermediate_validation_seconds,
              'process_seconds': time.perf_counter() - total_started,
              'torch_version': str(torch.__version__), 'threads': args.threads,
              'checkpoint_sha256': sha(checkpoint), 'implementation_sha256': implementation_sha,
              **device_metrics(device)}
    (args.run_dir / 'metrics.json').write_text(json.dumps(result, indent=2) + '\n')
    print(json.dumps(result | {'history': [], 'validation_history': []}, indent=2), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/code/suite.py
"""Run a batch of MP1 experiments described by a JSON spec.

Each experiment trains one model (provided ``train.py`` or our
``train_student.py``) into ``runs/<name>`` and records the validation BPB in a
summary table. Existing runs are skipped unless ``--force`` is given, so the
suite can be resumed across GPU sessions.

Spec format::

    {"experiments": [
        {"name": "student_eq", "trainer": "student", "implementation": "student",
         "config": "configs/student.json", "config_overrides": {"depth": 6},
         "steps": 1200, "batch_size": 32, "lr": 0.001, "warmup": 100,
         "min_lr_ratio": 0.1, "weight_decay": 0.1, "ema_decay": 0.999,
         "eval_every": 300, "seed": 17}
    ]}

Usage:
    python suite.py --spec experiments.json --device cuda
"""
import argparse
import csv
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
RUNS = ROOT / 'runs'
GENERATED = ROOT / 'configs/generated'


def build_config(exp):
    base = json.loads((ROOT / exp.get('config', 'configs/student.json')).read_text(encoding='utf-8-sig'))
    base.update(exp.get('config_overrides', {}))
    GENERATED.mkdir(parents=True, exist_ok=True)
    path = GENERATED / f"{exp['name']}.json"
    path.write_text(json.dumps(base, indent=2) + '\n')
    return path, base


def build_argv(exp, config_path, device, threads, runs_dir):
    if exp.get('trainer', 'student') == 'provided':
        argv = [sys.executable, 'train.py',
                '--implementation', exp.get('implementation', 'model'),
                '--config', str(config_path),
                '--run-dir', str(runs_dir / exp['name']),
                '--device', device, '--threads', str(threads),
                '--seed', str(exp.get('seed', 17)),
                '--steps', str(exp['steps']),
                '--batch-size', str(exp.get('batch_size', 32))]
        if exp.get('eval_every', 0):
            argv += ['--eval-every', str(exp['eval_every'])]
        return argv
    argv = [sys.executable, 'train_student.py',
            '--implementation', exp.get('implementation', 'student'),
            '--config', str(config_path),
            '--run-dir', str(runs_dir / exp['name']),
            '--device', device, '--threads', str(threads),
            '--seed', str(exp.get('seed', 17)),
            '--steps', str(exp['steps']),
            '--batch-size', str(exp.get('batch_size', 32)),
            '--lr', str(exp.get('lr', 1e-3)),
            '--min-lr-ratio', str(exp.get('min_lr_ratio', 0.1)),
            '--warmup', str(exp.get('warmup', 100)),
            '--weight-decay', str(exp.get('weight_decay', 0.1)),
            '--beta1', str(exp.get('beta1', 0.9)),
            '--beta2', str(exp.get('beta2', 0.95)),
            '--grad-clip', str(exp.get('grad_clip', 1.0)),
            '--ema-decay', str(exp.get('ema_decay', 0.999)),
            '--eval-every', str(exp.get('eval_every', 500)),
            '--log-every', str(exp.get('log_every', 100))]
    if exp.get('no_ema'):
        argv += ['--no-ema']
    if exp.get('compile'):
        argv += ['--compile']
    return argv


def summarize(exp, metrics):
    result = {'name': exp['name'], 'parameters': metrics.get('parameters'),
              'train_tokens': metrics.get('train_tokens'),
              'train_seconds': metrics.get('train_seconds'),
              'seed': metrics.get('seed')}
    if 'selected' in metrics:
        result['validation_bpb'] = metrics['selected']['bpb']
        result['selected_source'] = metrics['selected']['source']
    else:
        result['validation_bpb'] = metrics['validation']['bpb']
        result['selected_source'] = 'raw'
    return result


def write_summary(runs_dir, results):
    """Persist the summary after every experiment so a reset cannot lose it."""
    (runs_dir / 'summary.json').write_text(json.dumps(results, indent=2) + '\n')
    with (runs_dir / 'summary.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['name', 'validation_bpb', 'parameters',
                                                    'train_tokens', 'train_seconds', 'seed',
                                                    'selected_source'])
        writer.writeheader()
        for row in results:
            writer.writerow({k: row.get(k) for k in writer.fieldnames})


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--spec', type=Path, required=True)
    p.add_argument('--device', default='cuda')
    p.add_argument('--threads', type=int, default=4)
    p.add_argument('--only', default='', help='Comma separated experiment names.')
    p.add_argument('--runs-dir', type=Path, default=RUNS,
                   help='Where to store runs (keep it outside the code directory).')
    p.add_argument('--force', action='store_true')
    args = p.parse_args()

    experiments = json.loads(args.spec.read_text(encoding='utf-8-sig'))['experiments']
    wanted = {n.strip() for n in args.only.split(',') if n.strip()}
    if wanted:
        experiments = [e for e in experiments if e['name'] in wanted]
    runs_dir = args.runs_dir
    runs_dir.mkdir(parents=True, exist_ok=True)
    results = []
    for exp in experiments:
        run_dir = runs_dir / exp['name']
        metrics_path = run_dir / 'metrics.json'
        if metrics_path.exists() and not args.force:
            metrics = json.loads(metrics_path.read_text())
            row = summarize(exp, metrics)
            results.append(row)
            print(f"[skip] {exp['name']}: validation_bpb={row['validation_bpb']:.4f}", flush=True)
            continue
        config_path, config = build_config(exp)
        argv = build_argv(exp, config_path, args.device, args.threads, runs_dir)
        print(f"[run ] {exp['name']}: {' '.join(argv[1:])}", flush=True)
        log_path = runs_dir / f"{exp['name']}.log"
        with log_path.open('w') as log:
            proc = subprocess.run(argv, cwd=ROOT, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True)
            log.write(proc.stdout)
        if not metrics_path.exists():
            print(f"[fail] {exp['name']} produced no metrics.json (exit {proc.returncode}); see {log_path}", flush=True)
            results.append({'name': exp['name'], 'validation_bpb': None, 'error': proc.returncode})
            continue
        metrics = json.loads(metrics_path.read_text())
        row = summarize(exp, metrics)
        results.append(row)
        print(f"[done] {exp['name']}: validation_bpb={row['validation_bpb']:.4f} "
              f"params={row['parameters']} time={row['train_seconds']:.0f}s", flush=True)
        write_summary(runs_dir, results)

    write_summary(runs_dir, results)
    print('\n=== summary (lower BPB is better) ===', flush=True)
    for row in sorted(results, key=lambda r: (r.get('validation_bpb') is None, r.get('validation_bpb', 0))):
        bpb = row.get('validation_bpb')
        print(f"{row['name']:<28} {bpb if bpb is None else round(bpb, 4)}", flush=True)


if __name__ == '__main__':
    main()


In [ ]:
MODE = "full"   # "smoke" = quick check, "full" = experiment suite

import json, pathlib
SMOKE = [{'name': 'baseline_repro', 'trainer': 'provided', 'implementation': 'model', 'config': 'configs/baseline.json', 'steps': 1200, 'batch_size': 32, 'eval_every': 0, 'seed': 17}, {'name': 'student_smoke', 'trainer': 'student', 'implementation': 'student', 'config': 'configs/student.json', 'steps': 300, 'batch_size': 32, 'lr': 0.001, 'warmup': 50, 'min_lr_ratio': 0.1, 'eval_every': 300, 'seed': 17}]
FULL = [{'name': 'final_s42', 'trainer': 'student', 'implementation': 'student', 'config': 'configs/student.json', 'config_overrides': {'mlp': 'gelu', 'depth': 8, 'dropout': 0.2}, 'steps': 20000, 'batch_size': 32, 'lr': 0.001, 'warmup': 500, 'min_lr_ratio': 0.1, 'eval_every': 2000, 'seed': 42}, {'name': 'final_s123', 'trainer': 'student', 'implementation': 'student', 'config': 'configs/student.json', 'config_overrides': {'mlp': 'gelu', 'depth': 8, 'dropout': 0.2}, 'steps': 20000, 'batch_size': 32, 'lr': 0.001, 'warmup': 500, 'min_lr_ratio': 0.1, 'eval_every': 2000, 'seed': 123}]
pathlib.Path("configs").mkdir(exist_ok=True)
pathlib.Path("configs/student.json").write_text('{\n  "vocab": 2048,\n  "width": 256,\n  "heads": 8,\n  "depth": 6,\n  "context": 256,\n  "mlp_hidden": 768,\n  "dropout": 0.1,\n  "qk_norm": true,\n  "rope_base": 10000.0\n}\n')

experiments = {"smoke": SMOKE, "full": FULL}[MODE]
pathlib.Path("experiments.json").write_text(json.dumps({"experiments": experiments}, indent=2))
print("MODE =", MODE, "| experiments:", [e["name"] for e in experiments])


In [ ]:
!python suite.py --spec experiments.json --device cuda --threads 4 --runs-dir /kaggle/working/runs

In [ ]:
import shutil, pathlib
runs = pathlib.Path('/kaggle/working/runs')
print((runs / 'summary.csv').read_text())
shutil.make_archive('/kaggle/working/mp1_results', 'zip', str(runs))
for p in sorted(pathlib.Path('/kaggle/working').glob('mp1_results.zip')):
    print('saved', p, round(p.stat().st_size/1e6, 1), 'MB')